# 离线预览：人工答案，不是 Jev 实测

# TypeSafe 简介实验（Introduction Lab）

针对官方文档对应章节的可运行实验笔记，全部实验使用**中文场景与中文提示词**。
面向会基础 Python、刚接触 AI Agent 的读者。

**学习目标：** 区分 TypeSafe、Jev 与 System One，并将多个原子分数组合成可检查的结果。

[官方原文](https://docs.typesafe.ai/introduction) · [中文参考](https://bald0wang.github.io/jev-docs-zh/introduction/)。本章以中文重述理论、复刻对应场景；扩展实验会单独说明。
所有客户、订单及消息均为教学合成数据。

## 笔记本结构

| 章节 | 内容 |
|---|---|
| 0. 准备 | 安装库、配置客户端、连通性测试与离线示例 |
| 1 | 理论速览：请求、原语、概率、控制流 |
| 2 | 复刻创业路演的三个独立维度 |
| 练习与小结 | 练习、自查、总结与本次执行记录 |

实验按**原理 → 理论根基 → 定义数据 → 定义问题 → 调用 → 解读结果**展开，每个代码单元格只做一件事。

## 运行要求

- Python ≥ 3.10；本章使用 `typesafe-sdk==0.7.0`。
- 真实实验需要启动进程的 `TYPESAFE_API_KEY` 环境变量，密钥不要写进 Notebook。

在本仓库 `notebooks/` 目录创建环境并打开本文件：

```bash
./setup_env.sh
.venv/bin/python -m pip install -r requirements.txt -c generators/constraints-foundations.txt
.venv/bin/jupyter lab introduction_experiments.ipynb
```

产品名、字段名和选项 key 保持英文，state、提示词与解说使用中文。
默认 `JEV_RUN_MODE=live`，调用失败即停止；无密钥学习时，在启动 Jupyter 前设置 `JEV_RUN_MODE=offline`。
`auto` 仅供教学体验，缺密钥或 401 时显式回退；正式验收使用 `live`。

**验证状态：真实 API 待验收。** 本文件尚未执行真实 API；离线检查仅验证代码路径。
批量执行、离线预览和验收记录见本目录 `MAINTENANCE.md`。

## 0. 准备

本节可折叠阅读，但独立运行时不能跳过。客户端、辅助对象和示例数据都在本文件中定义。

### 0.1 安装依赖

推荐先运行 `setup_env.sh`。只有当前内核缺少 SDK 时，本格才安装依赖。

In [1]:
import importlib.util
if importlib.util.find_spec("typesafe_sdk") is None:
    %pip install -q typesafe-sdk==0.7.0

**观察与理解：** 安装包的名字是 typesafe-sdk，Python 导入名是 typesafe_sdk。安装成功不代表 API 已连通。

### 0.2 导入与配置

默认模型固定版本，便于记录实验条件；可通过环境变量更换。不要从 Notebook 输入密钥。

In [2]:
import os
import json
import time
import math
from datetime import datetime, timezone
from importlib.metadata import version
from typesafe_sdk import (
    Choice, Score, Noul, NoulCriteria, TypeSafeClient,
    TypeSafeAuthenticationError, RetryPolicy,
)

MODEL = os.environ.get("TYPESAFE_DEFAULT_MODEL", "jev-1.13.0")
RUN_MODE = os.environ.get("JEV_RUN_MODE", "live")
API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
if RUN_MODE not in {"live", "offline", "auto"}:
    raise ValueError("JEV_RUN_MODE 只能是 live、offline 或 auto")
if RUN_MODE == "live" and not API_KEY:
    raise RuntimeError("请在启动 Jupyter 前配置 TYPESAFE_API_KEY 环境变量")
client = None
if RUN_MODE != "offline" and API_KEY:
    client = TypeSafeClient(api_key=API_KEY, model=MODEL, timeout=30,
                           retry=RetryPolicy(max_retries=0))
print("模式：", RUN_MODE, "SDK：", version("typesafe-sdk"), "模型配置：", MODEL)

模式： offline SDK： 0.7.0 模型配置： jev-1.13.0


正式验收禁用自动回退，且不自动重试，以便请求数量有界。`auto` 与 `offline` 是教学工具，不代表成功连接模型。

### 0.3 连通性测试

用一条 Noul 检查真实响应能否返回。网络、限流与输入错误直接抛出，不伪装成不确定判断。

In [3]:
PING = {"source": "offline", "reason": "未发起连通性请求"}
if client is not None:
    try:
        ping = client.system_one("你好", {"greeting": Noul(
            instructions="这段文字是否在打招呼？")})
        PING = {"source": "live", "model": ping.model,
                "input_tokens": ping.usage.input_tokens,
                "output_tokens": ping.usage.output_tokens}
    except TypeSafeAuthenticationError:
        if RUN_MODE == "live":
            raise
        client.close()
        client = None
        PING["reason"] = "401 鉴权失败，仅教学模式允许回退"
print(json.dumps(PING, ensure_ascii=False))

{"source": "offline", "reason": "未发起连通性请求"}


**观察与理解：** source=live 表示这一次连通性请求成功；仍要查看后续实验记录，不能用它代替整章验收。

### 0.4 离线替身

沿用参考模板的 `_FakeAnswer` 与 `_FakeResponse` 访问方式。人工数字仅用来检验读取字段和代码分支。

In [4]:
class _FakeAnswer:
    def __init__(self, type_, **values):
        self.type = type_
        for name, value in values.items():
            setattr(self, name, value)


class _FakeResponse:
    def __init__(self, answers):
        self.answers = answers
        self.choices = {k: v for k, v in answers.items() if v.type == "choice"}
        self.scores = {k: v for k, v in answers.items() if v.type == "score"}
        self.nouls = {k: v for k, v in answers.items() if v.type == "noul"}
        self.model = "人工示例，非模型预测"
        self.usage = _FakeAnswer("usage", input_tokens=0, output_tokens=0)

人工 Score 由概率计算期望，避免模板中的分数与分布不一致。人工 confidence 只是指定的演示字段，不是在复现服务端的计算公式。

定义两种示例答案构造器；Noul 可直接用 `_FakeAnswer`。所有具体答案集中在下一节。

In [5]:
def fake_choice(probabilities, confidence):
    return _FakeAnswer("choice", choice=max(probabilities, key=probabilities.get),
                       probabilities=probabilities, confidence=confidence)


def fake_score(probabilities, legend, confidence):
    return _FakeAnswer("score", score=sum(k * p for k, p in probabilities.items()),
                       probabilities=probabilities, legend=dict(enumerate(legend)),
                       confidence=confidence)

**观察与理解：** 例如概率 {0:0.05, 1:0.26, 2:0.69} 对应 1.64。不能把另一个数与该分布配在一起。

### 0.5 统一调用入口

每次调用记录来源、模型与 token 用量。离线耗时记为 None，不把本地字典访问当成模型速度。

In [6]:
CALL_LOG = []


class TS:
    def call(self, state, questions, offline_answers, label):
        start = time.perf_counter()
        source = "live"
        if client is None:
            response, source = _FakeResponse(offline_answers), "offline"
        else:
            try:
                response = client.system_one(state, questions)
            except TypeSafeAuthenticationError:
                if RUN_MODE != "auto":
                    raise
                response, source = _FakeResponse(offline_answers), "offline"
        validate_response(response, questions)
        CALL_LOG.append({"case": label, "source": source, "model": response.model,
                         "seconds": time.perf_counter() - start if source == "live" else None,
                         "input_tokens": response.usage.input_tokens,
                         "output_tokens": response.usage.output_tokens})
        if source == "offline":
            print("离线示例：", label, "；人工答案，不是 Jev 实测")
        return response


ts = TS()

与参考模板相比，这里增加了严格 live 模式和逐次记录。保留 401 教学回退，但超时、429 等错误继续失败，防止验收被回退掩盖。

校验结构与数值契约；只断言接口应满足的性质，不断言真实模型必须预测某个标签。

In [7]:
def validate_response(response, questions):
    if set(response.answers) != set(questions):
        raise ValueError("答案 ID 与问题 ID 不一致")
    for key, question in questions.items():
        answer = response.answers[key]
        if isinstance(question, Noul):
            if not 0 <= answer.noul <= 1:
                raise ValueError("Noul 超出概率范围")
            continue
        probabilities = answer.probabilities
        if not all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities.values()):
            raise ValueError("概率值无效")
        if not math.isclose(sum(probabilities.values()), 1, abs_tol=0.02):
            raise ValueError("概率之和偏离 1")
        if not 0 <= answer.confidence <= 1:
            raise ValueError("confidence 超出范围")
        if isinstance(question, Choice):
            if set(probabilities) != set(question.criteria):
                raise ValueError("Choice 选项集合不一致")
            if answer.choice not in probabilities:
                raise ValueError("Choice 标签不在选项中")
        else:
            expected = sum(int(k) * p for k, p in probabilities.items())
            if not math.isclose(answer.score, expected, abs_tol=0.03):
                raise ValueError("Score 与概率加权期望不一致")

**观察与理解：** 容差用于服务端数值舍入。结构检查通过只说明响应可读取，不证明语义判断正确。

显示结果时统一列出类型、概率和置信度；Noul 不额外制造 confidence 字段。

In [8]:
def show(response):
    rows = {}
    for key, answer in response.answers.items():
        rows[key] = {name: getattr(answer, name) for name in
                     ("type", "choice", "score", "noul", "confidence", "probabilities", "legend")
                     if hasattr(answer, name)}
    print(json.dumps(rows, ensure_ascii=False, indent=2))

### 0.6 本章离线示例数据

以下数值全部人工构造，专门测试分支；不来自 Jev，也不能用于估计中文准确率或校准情况。正式 live 运行不会使用这些答案。

In [9]:
PITCH_OFFLINE = {
    "market_size": fake_score({0: 0.1, 1: 0.3, 2: 0.6},
        ["需求范围有限", "需求有一定规模", "广泛且明确的需求"], 0.71),
    "technical_feasibility": fake_score({0: 0.05, 1: 0.15, 2: 0.8},
        ["关键能力尚不可行", "有部分实现证据", "主要能力已可运行"], 0.83),
    "differentiation": fake_score({0: 0.4, 1: 0.5, 2: 0.1},
        ["与已有方案接近", "局部差异", "有明确且难替代的差异"], 0.57),
}

## 1. 📖 理论速览

TypeSafe 是提供 API 与 SDK 的平台；Jev 是其模型；System One 是文档给这类聚焦判断模型的名称。
请求送入 `state + questions`，返回类型化答案；Python 根据答案执行 `if`、排序或路由。
它适合处理“归哪一类”“符合条件吗”“量表几分”，不负责生成长回复、代码或思维过程。

“一秒判断”是设计问题时的尺度：给一位了解上下文的人，能否迅速判断一个明确属性？
这不是本教程对 API 时延的承诺。复杂任务应拆解，已有确定性规则继续由代码执行。

### 状态、问题和答案的关系

`state` 是待判断材料，可用字符串、对象或文本数组；本系列聚焦文本，不直接传图像、音频或视频。
`instructions` 写完整判断任务；`criteria` 描述可选答案；问题 ID 供代码取回答案。

同一次请求的所有问题看到同一个 state，并独立评估。一个问题不能引用同请求另一个问题尚未返回的答案。
若确实有依赖，先在代码中取回结果，再决定是否构造下一次请求。

问题 ID 在 HTTP 请求中用于映射，但不会作为语义提示发送给模型；例如 `refund_requested` 这个名字不能代替完整 instructions。
此规则不表示 state 中的订单号也会被隐藏。

| 原语 | 适合的判断 | 主要返回值 |
|---|---|---|
| Choice | 从定义好的选项中选一项 | choice、probabilities、confidence |
| Score | 按有序等级评价一个属性 | score、legend、probabilities、confidence |
| Noul | 一个明确命题是否成立 | noul，范围 0–1；没有独立 confidence |


[官方原文](https://docs.typesafe.ai/concepts/state) · [中文参考](https://bald0wang.github.io/jev-docs-zh/concepts/state/) · [官方原文](https://docs.typesafe.ai/primitives) · [中文参考](https://bald0wang.github.io/jev-docs-zh/primitives/)

### 概率和 confidence 怎样读

Score 是等级的概率加权期望，所以可以有小数：`0×0 + 1×0.57 + 2×0.43 = 1.43`。
Choice 的低 confidence 往往提示选项之间难以区分；Score 的不确定性还与概率落在相邻还是相距较远的等级有关。
confidence 是分布信息的压缩摘要，不能统一当成最大类别概率。

校准描述一组预测中概率与实际发生频率的关系。模型给某件事 0.8 的概率，不保证这件事一定发生；
高 confidence 的个例也可能错。“我不知道”或转人工是有用的业务出口。
对中文任务要在自己的样本上验证，不承诺中文与英语效果相同。


[官方原文](https://docs.typesafe.ai/confidence) · [中文参考](https://bald0wang.github.io/jev-docs-zh/confidence/) · [官方原文](https://docs.typesafe.ai/models) · [中文参考](https://bald0wang.github.io/jev-docs-zh/models/)

### 谁决定下一步

| 架构 | 判断与控制流怎样分工 | 适用提醒 |
|---|---|---|
| 传统软件 | 规则与分支都由代码定义 | 可计算的条件直接写规则 |
| LLM 智能体 | 模型参与选择下一步动作 | 需要适当的工具约束与监督 |
| AI 驱动的软件 | 代码定流程，模型回答狭窄问题 | 本教程采用这种结构 |

这是参考文档的架构划分，实际系统可以混合使用。拥有类型化输出，不代表已经验证业务正确性。


[官方原文](https://docs.typesafe.ai/concepts/how-to-build-with-system-one) · [中文参考](https://bald0wang.github.io/jev-docs-zh/concepts/how-to-build-with-system-one/)

## 2. 配方：把创业路演拆成三个维度

原理：市场、技术和差异化分别判断，代码决定权重。对应简介中的创业路演示例，以下路演正文为中文教学补全。

### 📖 理论根基

“给路演一个总分”隐藏了多个偏好。拆开后，读者可以查看哪个因素拉低分数，也可以在不重新请求模型的情况下调整权重。
本例不构成投资判断；只是展示可组合的有序量表。

### 第一步：准备材料

把已有事实写进 state，不把期望答案混进去。

In [10]:
PITCH = {
    "product": "为小型连锁门店提供库存提醒的软件",
    "market": "访谈了二十家门店，其中十二家报告缺货或积压问题；尚无更大样本",
    "technology": "已有能连接两种收银系统的原型，并在两家门店试用",
    "competition": "市场已有同类软件，目前主要差别是部署较方便",
}

### 第二步：定义独立量表

每个 Score 只判断一个维度，三个量表都使用 0–2，便于演示归一化。

In [11]:
PITCH_QUESTIONS = {
    "market_size": Score(instructions="仅据 market 中的证据，评价需求范围。",
        criteria=["需求范围有限", "需求有一定规模", "广泛且明确的需求"]),
    "technical_feasibility": Score(instructions="仅据 technology，评价当前技术可行性。",
        criteria=["关键能力尚不可行", "有部分实现证据", "主要能力已可运行"]),
    "differentiation": Score(instructions="仅据 competition，评价与现有方案的差异。",
        criteria=["与已有方案接近", "局部差异", "有明确且难替代的差异"]),
}

**观察与理解：** 把问题写在 instructions 中。用 question ID 取结果，不靠名称暗示评判标准。

### 第三步：一次请求三个判断

In [12]:
pitch_response = ts.call(PITCH, PITCH_QUESTIONS, PITCH_OFFLINE, "创业路演三维评分")

离线示例： 创业路演三维评分 ；人工答案，不是 Jev 实测


### 第四步：先看分布，再看总分

In [13]:
show(pitch_response)

{
  "market_size": {
    "type": "score",
    "score": 1.5,
    "confidence": 0.71,
    "probabilities": {
      "0": 0.1,
      "1": 0.3,
      "2": 0.6
    },
    "legend": {
      "0": "需求范围有限",
      "1": "需求有一定规模",
      "2": "广泛且明确的需求"
    }
  },
  "technical_feasibility": {
    "type": "score",
    "score": 1.75,
    "confidence": 0.83,
    "probabilities": {
      "0": 0.05,
      "1": 0.15,
      "2": 0.8
    },
    "legend": {
      "0": "关键能力尚不可行",
      "1": "有部分实现证据",
      "2": "主要能力已可运行"
    }
  },
  "differentiation": {
    "type": "score",
    "score": 0.7,
    "confidence": 0.57,
    "probabilities": {
      "0": 0.4,
      "1": 0.5,
      "2": 0.1
    },
    "legend": {
      "0": "与已有方案接近",
      "1": "局部差异",
      "2": "有明确且难替代的差异"
    }
  }
}


**观察与理解：** 真实分数可能与人工预览不同。阅读每个分布，检查它是否支持你对该维度的解读。

### 第五步：代码决定权重

同一批模型输出，比较两套教学权重；权重总和为 1，Score 先除以最高档 2。

In [14]:
WEIGHT_SETS = {
    "偏重需求": {"market_size": 0.5, "technical_feasibility": 0.3, "differentiation": 0.2},
    "偏重技术": {"market_size": 0.2, "technical_feasibility": 0.6, "differentiation": 0.2},
}
combined = {
    name: sum(weights[key] * pitch_response.scores[key].score / 2 for key in weights)
    for name, weights in WEIGHT_SETS.items()
}

显示组合结果。它是应用自行定义的指标，不是模型返回的成功概率。

In [15]:
print(json.dumps(combined, ensure_ascii=False, indent=2))

{
  "偏重需求": 0.7075,
  "偏重技术": 0.745
}


**观察与理解：** 权重变化不保证排序反转。先记录实际变化，再解释变化来源；不要为得到漂亮结论篡改返回值。

## 练习与自查

把‘文章质量好不好’拆成三个互不依赖的问题，并注明哪些标准可以由代码直接检查。

<details><summary>参考思路：先完成练习再展开</summary>

可以分别评价是否回应主题、论据是否支持结论、语言是否清楚；字数与链接是否为空可以先由代码检查。

</details>

## 小结

| 学到的接口 | 使用方式 |
|---|---|
| state | 提供事实 |
| questions | 定义原子判断 |
| typed answers | 交给代码检查和组合 |

下一章：[快速开始](quickstart_experiments.ipynb)。详细原语与架构模式由对应作者的章节继续展开。

离线运行只说明教材代码能执行。正式交付必须实际运行 live，并阅读每条输出；缺失的分支应记为未观察到。

## 本次执行记录

先关闭连接，再生成记录。下面的 JSON 由实际运行计算，批量执行器会据此检查来源。

In [16]:
if client is not None:
    client.close()

真实探针只演示行为路径；若据其返回挑选样例，这批样例就不适合再当作无偏准确率测试集。延迟也只是本次网络环境中的观测。

In [17]:
AUDIT = {
    "kind": "jev_execution_audit",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "sdk": version("typesafe-sdk"), "requested_model": MODEL,
    "mode": RUN_MODE, "ping": PING,
    "real_calls": sum(x["source"] == "live" for x in CALL_LOG),
    "offline_calls": sum(x["source"] == "offline" for x in CALL_LOG),
    "cases": CALL_LOG,
    "coverage": globals().get("COVERAGE", {}),
    "validation_status": "live_executed_requires_review" if (
        PING["source"] == "live" and CALL_LOG
        and all(x["source"] == "live" for x in CALL_LOG)
    ) else "offline_only_not_model_evidence",
}
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))

{
  "kind": "jev_execution_audit",
  "executed_at_utc": "2026-09-23T15:36:30.945708+00:00",
  "sdk": "0.7.0",
  "requested_model": "jev-1.13.0",
  "mode": "offline",
  "ping": {
    "source": "offline",
    "reason": "未发起连通性请求"
  },
  "real_calls": 0,
  "offline_calls": 1,
  "cases": [
    {
      "case": "创业路演三维评分",
      "source": "offline",
      "model": "人工示例，非模型预测",
      "seconds": null,
      "input_tokens": 0,
      "output_tokens": 0
    }
  ],
  "coverage": {},
  "validation_status": "offline_only_not_model_evidence"
}


读完输出后，在本仓库 `notebooks/MAINTENANCE.md` 的验收表中记录日期、真实模型、观察到的分支和偏离预期之处。不要把人工演示数值抄进实测记录。